# Traffic Data Visualization

This notebook visualizes traffic speed data from the Traffic API using MapboxGL.

In [ ]:
# Install required packages
!pip install mapboxgl jupyter pandas requests -q

In [ ]:
import requests
import json

## Configuration

In [ ]:
# API endpoint
API_BASE = "http://localhost:8000"

# Mapbox token - REPLACE WITH YOUR TOKEN
MAPBOX_TOKEN = "YOUR_MAPBOX_TOKEN_HERE"

## Fetch Data from API

In [ ]:
def fetch_aggregates(day, period):
    """Fetch aggregated speed data for a day and period."""
    url = f"{API_BASE}/aggregates/"
    params = {"day": day, "period": period}
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()


def fetch_spatial_filter(day, period, bbox):
    """Fetch road segments within a bounding box."""
    url = f"{API_BASE}/aggregates/spatial_filter/"
    payload = {"day": day, "period": period, "bbox": bbox}
    response = requests.post(url, json=payload)
    response.raise_for_status()
    return response.json()


def fetch_slow_links(period, threshold, min_days):
    """Fetch slow links based on threshold."""
    url = f"{API_BASE}/patterns/slow_links/"
    params = {"period": period, "threshold": threshold, "min_days": min_days}
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

## Test API Endpoints

In [ ]:
# Test aggregates endpoint
print("Testing /aggregates/ endpoint...")
aggregates = fetch_aggregates("Wednesday", "AM Peak")
print(f"Retrieved {len(aggregates)} links")
print("Sample:", aggregates[0] if aggregates else "No data")

In [ ]:
# Test spatial filter endpoint
print("Testing /aggregates/spatial_filter/ endpoint...")
bbox = [-81.8, 30.1, -81.6, 30.3]
spatial = fetch_spatial_filter("Wednesday", "AM Peak", bbox)
print(f"Retrieved {len(spatial)} segments in bbox {bbox}")
print("Sample:", spatial[0] if spatial else "No data")

In [ ]:
# Test slow links endpoint
print("Testing /patterns/slow_links/ endpoint...")
slow_links = fetch_slow_links("PM Peak", 30.0, 3)
print(f"Retrieved {len(slow_links)} slow links")
print("Sample:", slow_links[0] if slow_links else "No data")

## Visualize with MapboxGL

In [ ]:
from mapboxgl import Map

# Create map
m = Map(
    access_token=MAPBOX_TOKEN,
    style="mapbox://styles/mapbox/streets-v11",
    center=[-81.7, 30.2],  # Jacksonville, FL area
    zoom=10,
)
m

In [ ]:
# Get spatial data with geometry
spatial_data = fetch_spatial_filter("Wednesday", "AM Peak", [-81.8, 30.1, -81.6, 30.3])
print(f"Got {len(spatial_data)} road segments")

In [ ]:
# Convert GeoJSON to mapboxgl format
geojson_features = []
for item in spatial_data:
    if item.get("geometry"):
        feature = {
            "type": "Feature",
            "properties": {
                "link_id": item["link_id"],
                "name": item.get("name", "Unknown"),
                "avg_speed": item["avg_speed"],
            },
            "geometry": json.loads(item["geometry"]),
        }
        geojson_features.append(feature)

geojson_collection = {"type": "FeatureCollection", "features": geojson_features}

print(f"Created GeoJSON with {len(geojson_features)} features")

In [ ]:
# Add geojson layer to map with color by speed
from mapboxgl.utils import create_color_stops

# Color stops based on average speed (green = fast, red = slow)
color_stops = create_color_stops([0, 20, 40, 60, 80], colors=["#d73027", "#fc8d59", "#fee08b", "#d9ef8b", "#1a9850"])

m.add_layer(
    {
        "id": "traffic-speed",
        "type": "line",
        "source": {"type": "geojson", "data": geojson_collection},
        "layout": {"line-color": "#case", "line-width": 3},
        "paint": {
            "line-color": [
                "interpolate",
                ["linear"],
                ["get", "avg_speed"],
                *[
                    [speed, color]
                    for speed, color in zip(
                        [0, 20, 40, 60, 80],
                        ["#d73027", "#fc8d59", "#fee08b", "#d9ef8b", "#1a9850"],
                    )
                ],
            ],
            "line-width": 4,
        },
    }
)

In [ ]:
m

## Alternative: Simple Visualization with Folium

In [ ]:
# Install folium for alternative visualization
!pip install folium -q

In [ ]:
import folium

# Create a simple map
m2 = folium.Map(location=[30.2, -81.7], zoom_start=10, tiles="cartodbpositron")

# Add geojson layer
folium.GeoJson(
    geojson_collection,
    name="traffic_speed",
    style_function=lambda x: {
        "color": "red"
        if x["properties"]["avg_speed"] < 20
        else "orange"
        if x["properties"]["avg_speed"] < 40
        else "green"
        if x["properties"]["avg_speed"] >= 60
        else "yellow",
        "weight": 3,
    },
).add_to(m2)

folium.LayerControl().add_to(m2)
m2

## Summary

This notebook demonstrates:
1. Fetching data from the Traffic API
2. Visualizing road segments with speed data using MapboxGL
3. Alternative visualization with Folium

To use MapboxGL properly, replace `YOUR_MAPBOX_TOKEN_HERE` with your actual Mapbox access token.